# 005 · None

> **Syntax › Data Types** · Python 3.14

`None` is Python's "no value" — the counterpart of C#'s `null`. The big differences: `None` is a real **object** (the single instance of `NoneType`), you test for it with `is`, and Python has no `?.` or `??` operators, so you use a few idioms instead.

**In this notebook you will learn:**

- what `None` is and why it is a singleton
- why you write `x is None` rather than `x == None`
- that functions without a `return` give back `None`
- how to type-hint optional values (`str | None`) vs C#'s `string?`
- idioms that replace `?.` and `??` — and the `or` pitfall

## C# mental model

| C# | Python |
|---|---|
| `null` | `None` |
| `x == null` / `x is null` | `x is None` |
| `x != null` / `x is not null` | `x is not None` |
| `string?` (nullable reference type) | `str \| None` (type hint) |
| `int?` / `Nullable<int>` | `int \| None` (no separate wrapper type) |
| `x ?? fallback` | `x if x is not None else fallback` |
| `x?.Name` | `x.name if x is not None else None` |
| `void` method | function returning `None` |
| `NullReferenceException` | `AttributeError` / `TypeError` on `None` |

## `None` is an object

In C#, `null` is the absence of a reference. In Python, `None` is an actual object — the one and only instance of the class `NoneType`. Because there is exactly one, every `None` in a program is the same object.

In [ ]:
x = None
print(x)              # None
print(type(x))        # <class 'NoneType'>
print(repr(x))        # None

a = None
b = None
print(a is b)         # True — same object
print(id(a) == id(None))

`NoneType` is available as `types.NoneType` (Python 3.10+), and you can't create another instance — calling it just returns the singleton:

In [ ]:
import types

print(types.NoneType is type(None))   # True
print(type(None)() is None)           # True — still the singleton

## Testing for `None`: use `is`

PEP 8 says comparisons to singletons like `None` should **always** use `is` or `is not`, never `==`. Two reasons:

1. `is` checks identity and cannot be overridden; `==` calls `__eq__`, which a class can redefine.
2. It states intent clearly — similar to C#'s `x is null` pattern, which also ignores overloaded `==`.

```csharp
// C#
if (customer is null) return;
if (customer is not null) Console.WriteLine(customer.Name);
```

In [ ]:
customer = None

if customer is None:
    print("no customer")

customer = "alice"
if customer is not None:
    print(f"customer: {customer}")

⚠️ **Gotcha:** an object with a custom `__eq__` can claim to equal `None`. `is` is never fooled:

In [ ]:
class Weird:
    def __eq__(self, other):
        return True          # equal to everything!

w = Weird()
print(w == None)     # True  — misleading
print(w is None)     # False — correct

## Functions return `None` implicitly

There is no `void` in Python. A function without a `return` statement — or with a bare `return` — returns `None`.

```csharp
// C#
void Greet(string name) => Console.WriteLine($"Hi {name}");
// var r = Greet("bob");   // compile error: can't assign void
```

In [ ]:
def greet(name: str) -> None:      # '-> None' is the Python way to say 'void'
    print(f"Hi {name}")

result = greet("bob")
print(result)                      # None — no compile error, you just get None

⚠️ **Gotcha:** many mutating methods return `None` — `list.sort()`, `list.append()`, `dict.update()`. Assigning their result is a common bug:

In [ ]:
nums = [3, 1, 2]
nums = nums.sort()       # BUG: sort() sorts in place and returns None
print(nums)              # None

## `None` as "not found"

Many standard APIs return `None` to signal "missing" instead of throwing — like C#'s `TryGetValue` / `FirstOrDefault`. `dict.get` is the classic example:

In [ ]:
prices = {"pen": 1.20, "cup": 4.50}

print(prices.get("pen"))          # 1.2
print(prices.get("lamp"))         # None   — no KeyError
print(prices.get("lamp", 0.0))    # 0.0    — default instead of None

import re
match = re.search(r"\d+", "no digits here")
print(match)                      # None — always check before using it

## Optional type hints: `str | None`

C# 8+ nullable reference types make `string?` explicit and the compiler warns on possible null dereferences. Python's equivalent is a **type hint**: `str | None` (or the older `Optional[str]` from `typing`). It is checked by tools like mypy/pyright — not at runtime. *(Type hints are covered in Type Hints › 001 · Type Annotations.)*

```csharp
// C#
string? FindUser(int id) => id == 1 ? "alice" : null;
```

In [ ]:
def find_user(user_id: int) -> str | None:
    return "alice" if user_id == 1 else None

user = find_user(2)
print(user)

# A type checker would flag `user.upper()` here: user might be None.
if user is not None:
    print(user.upper())    # narrowed to `str` inside this block

Calling a method on `None` fails at runtime with `AttributeError` — Python's version of `NullReferenceException`:

In [ ]:
user = find_user(2)
try:
    user.upper()
except AttributeError as e:
    print(f"{type(e).__name__}: {e}")

## No `?.` or `??` — idioms instead

Python has no null-conditional or null-coalescing operators (PEP 505 proposed them but was deferred). Use these idioms instead.

### Replacing `??`

```csharp
// C#
var display = nickname ?? "guest";
```

The precise equivalent is a conditional expression with `is not None`:

In [ ]:
nickname = None
display = nickname if nickname is not None else "guest"
print(display)       # guest

You will often see `x or default`. It is shorter, but it replaces **every falsy value**, not just `None`:

⚠️ **Gotcha:** `or` treats `0`, `""`, `[]`, and `False` like `None`.

In [ ]:
def set_volume(level: int | None = None) -> int:
    return level or 50                          # BUG: level=0 becomes 50

def set_volume_fixed(level: int | None = None) -> int:
    return level if level is not None else 50   # behaves like C#'s ??

print(set_volume(0), set_volume_fixed(0))       # 50 0
print(set_volume(), set_volume_fixed())         # 50 50

Use `or` only when every falsy value genuinely should get the default (e.g. treating an empty string like a missing one).

### Replacing `?.`

```csharp
// C#
var city = order?.Customer?.Address?.City;
```

For one level, a conditional expression works. For attribute chains, `getattr(obj, "name", default)` returns the default when the attribute is missing — and `getattr(None, "x", None)` is `None`, so it chains safely:

In [ ]:
class Address:
    def __init__(self, city: str):
        self.city = city

class Customer:
    def __init__(self, address: Address | None):
        self.address = address

c1 = Customer(Address("Springfield"))
c2 = Customer(None)

for c in (c1, c2):
    city = c.address.city if c.address is not None else None   # ≈ c.Address?.City
    print(city)

In [ ]:
def city_of(customer):
    return getattr(getattr(customer, "address", None), "city", None)

print(city_of(c1), city_of(c2), city_of(None))   # Springfield None None

For dictionaries, chained `.get()` calls play the role of `?.`:

In [ ]:
order = {"customer": {"address": None}}

city = (order.get("customer") or {}).get("address") or {}
print(city.get("city", "unknown"))    # unknown

## `None` as a default parameter

`None` is the conventional "not supplied" marker for parameters — especially for mutable defaults, which must never be written as `def f(items=[])`. *(Details in Functions › 002 · Parameters and Arguments.)*

In [ ]:
def add_tag(tag: str, tags: list[str] | None = None) -> list[str]:
    if tags is None:
        tags = []            # fresh list on every call
    tags.append(tag)
    return tags

print(add_tag("new"))
print(add_tag("sale"))       # independent — not ['new', 'sale']

## `None` is falsy, but not equal to `False` or `0`

In [ ]:
print(bool(None))                # False
print(None == False, None == 0)  # False False
print(None is False)             # False

try:
    None < 1
except TypeError as e:
    print(f"{type(e).__name__}: {e}")   # no ordering with None (C#: null < 1 is just false for int?)

## Summary

| Task | C# | Python |
|---|---|---|
| No value | `null` | `None` |
| Null check | `x is null` | `x is None` |
| Not-null check | `x is not null` | `x is not None` |
| Nullable type | `string?` / `int?` | `str \| None` / `int \| None` |
| Coalesce | `x ?? d` | `x if x is not None else d` |
| Coalesce (any falsy) | — | `x or d` |
| Null-conditional | `x?.Prop` | `x.prop if x is not None else None` / `getattr(x, "prop", None)` |
| Dict lookup or null | `d.GetValueOrDefault(k)` | `d.get(k)` |
| Void return | `void` | `-> None` |
| Null deref error | `NullReferenceException` | `AttributeError` |

## Exercises

1. Write `first_or_none(items)` that returns the first element of a list, or `None` if it is empty (like LINQ `FirstOrDefault`).
2. Write `coalesce(*values)` that returns the first argument that is **not `None`** (like chaining `a ?? b ?? c`). `coalesce(None, 0, 5)` must return `0`.
3. Given `config = {"db": {"host": "localhost"}}`, write a one-liner that reads `config["db"]["port"]` and defaults to `5432` when either level is missing.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
def first_or_none(items):
    return items[0] if items else None

print(first_or_none([]), first_or_none([7, 8]))   # None 7

# 2
def coalesce(*values):
    return next((v for v in values if v is not None), None)

print(coalesce(None, 0, 5))     # 0 — 0 is not None, so it wins
print(coalesce(None, None))     # None

# 3
config = {"db": {"host": "localhost"}}
port = config.get("db", {}).get("port", 5432)
print(port)   # 5432
```

</details>